# Round-Robin Conversation: Research Brainstorming Team

This notebook demonstrates autonomous multi-agent orchestration using a round-robin pattern.

Scenario:

Four agents brainstorm a research idea:

- Literature Expert
- Methodology Expert
- Critical Reviewer
- Implementation Expert

They speak in a fixed repeating order.

The collaboration stops when either the maximum message limit is reached or an agent writes `FINAL IDEA`.

## Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)



from picoagents.orchestration import RoundRobinOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

API key loaded successfully.


## Create specialized agents

In [4]:
class BrainstormTaskInput(BaseModel):
    """Input model holding the research brainstorming prompt."""
    topic: str


def create_specialized_agents(model_client: OpenAIChatCompletionClient) -> List[Agent]:
    """Create specialist agents in the fixed round-robin speaking order."""
    literature_expert = Agent(
        name="literature_expert",
        instructions=(
            "You are a literature expert. "
            "Identify relevant theory, research gaps, and possible citations. "
            "Be concise."
        ),
        model_client=model_client
    )

    methodology_expert = Agent(
        name="methodology_expert",
        instructions=(
            "You are a methodology expert. "
            "Propose study design, data collection, and evaluation methods. "
            "Be concise."
        ),
        model_client=model_client
    )

    critical_reviewer = Agent(
        name="critical_reviewer",
        instructions=(
            "You are a critical reviewer. "
            "Identify weaknesses, risks, confounds, and missing evidence. "
            "Be constructive and concise."
        ),
        model_client=model_client
    )

    implementation_expert = Agent(
        name="implementation_expert",
        instructions=(
            "You are an implementation expert. "
            "Suggest tools, datasets, and practical implementation steps. "
            "If the idea is sufficiently clear, end your response with 'FINAL IDEA'."
        ),
        model_client=model_client
    )

    return [
        literature_expert,
        methodology_expert,
        critical_reviewer,
        implementation_expert,
    ]


specialized_agents = create_specialized_agents(client)
literature_expert, methodology_expert, critical_reviewer, implementation_expert = specialized_agents

## Configure and run the round-robin orchestrator

### Conversation Flow Diagram

The orchestrator rotates through agents in a fixed order.
After each turn, termination is checked.

```mermaid
flowchart
    A[literature_expert] --> B[methodology_expert]
    B --> C[critical_reviewer]
    C --> D[implementation_expert]
    D --> A
    A -. check stop .-> T{Termination?}
    B -. check stop .-> T
    C -. check stop .-> T
    D -. check stop .-> T
    T -->|MaxMessageTermination(8) or FINAL IDEA| Z[End conversation]
```

Why this is predictable:
- Speaking order is fixed by the agents list.
- Stop conditions are explicit and checked every turn.

In [5]:
def build_brainstorming_orchestrator(agents: List[Agent]) -> RoundRobinOrchestrator:
    """Configure round-robin turn-taking and termination conditions."""
    return RoundRobinOrchestrator(
        agents=agents,
        termination=(
            MaxMessageTermination(8) |
            TextMentionTermination("FINAL IDEA")
        ),
    )


brainstorming_orchestrator = build_brainstorming_orchestrator(specialized_agents)

task_input = BrainstormTaskInput(
    topic=(
        "Brainstorm a master's thesis topic about using webcam-based eye tracking "
        "to support reading-comprehension research in online learning."
    )
)

async for message in brainstorming_orchestrator.run_stream(task_input.topic):
    print(message)

[user] 21:12:55 | Brainstorm a master's thesis topic about using webcam-based eye tracking to support reading-comprehension research in online learning.
[literature_expert] 21:13:27 | Concise thesis framing
- Use webcam-based eye tracking to measure readers’ visual attention and infer reading comprehension in authentic online-learning tasks (e.g., course pages, e‑texts, quizzes), with validation vs lab eye-trackers and exploration of real-time/adaptive supports.

Relevant theory (short)
- Eye–mind hypothesis (Just & Carpenter, 1980): fixation corresponds to cognitive processing.
- Models of eye-movement control in reading: E‑Z Reader (Reichle et al., 1998) and SWIFT (Engbert et al., 2005) — guide expectations about fixation durations, regressions, word-level measures.
- Cognitive Load Theory (Sweller): links processing demands to observable reading behavior (longer fixations, regressions).
- Information Foraging (Pirolli & Card, 1999) and hypertext/document-use theories (Rouet): readin

## Reflection questions

1. How is this different from a workflow graph?
2. Why is this pattern still predictable compared with AI-driven speaker selection?
3. What risks remain even with round-robin turn-taking?